# AS03 Summarize data with Pandas

## Download data

到內政部開放資料平台上或開放資料平台尋找並下載[「結婚對數按婚姻類型、性別及年齡分(按發生)」](https://data.gov.tw/dataset/130544)資料（CSV）並讀取。或者可以用API讀取`https://od.moi.gov.tw/api/v1/rest/datastore/301000000A-001660-015`。


### Read from Web API (JSON)

In [45]:
import requests
import json
import pandas as pd

url = "https://od.moi.gov.tw/api/v1/rest/datastore/301000000A-001660-015"
res = requests.get(url).json()
# print(res)
df = pd.DataFrame(res['result']['records'])
df = df.iloc[1: , :]
df
# df = pd.read_json(url)
# df

,statistic_yyy,according,site_id,marriage_type,female_age_or_spouse1,male_age_or_spouse2,marry_pair
1,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,未滿15歲,0
2,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,15歲,0
3,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,16歲,0
4,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,17歲,0
5,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,18歲,0
...,...,...,...,...,...,...,...
1995,110,按發生日期分,新北市板橋區,相同性別_女性,55～64歲,32歲,0
1996,110,按發生日期分,新北市板橋區,相同性別_女性,55～64歲,33歲,0
1997,110,按發生日期分,新北市板橋區,相同性別_女性,55～64歲,34歲,0
1998,110,按發生日期分,新北市板橋區,相同性別_女性,55～64歲,35～39歲,0


注意觀察資料，這個資料是，某鄉鎮市區（`site_id`）的某一種結婚型態（`marriage_type`）的雙方年齡組（`female_age_or_spouse1`, `male_age_or_spouse2`）有幾對（`marry_pair`）。如果把`marry_pair`全部相加，就會是當年度婚姻的總對數。

### Read CSV from web path

In [22]:
csv_path = "https://data.moi.gov.tw/MoiOD/System/DownloadFile.aspx?DATA=D6E64661-DD38-4244-A86B-396F5F00808B"
df = pd.read_csv(csv_path, skiprows = [1])
df.head()

,statistic_yyy,according,site_id,marriage_type,female_age_or_spouse1,male_age_or_spouse2,marry_pair
0,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,未滿15歲,0
1,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,15歲,0
2,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,16歲,0
3,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,17歲,0
4,110,按發生日期分,新北市板橋區,不同性別,未滿15歲,18歲,0


### Read local CSV

In [15]:
# Modify the path according to your PC
# local_path = "../../data/opendata108m040.csv"
# df = pd.read_csv(local_path, skiprows = [1])
# df.head()

## 1. Data Description

In [26]:
# 列印出所有變數
print(df.columns)

# 列印出所有變項的變數型態
# 注意marry_pair這個變項是否為整數，如果不是整數的話，為了後續運算，你需要把它轉整數
df.dtypes

Index(['statistic_yyy', 'according', 'site_id', 'marriage_type',
       'female_age_or_spouse1', 'male_age_or_spouse2', 'marry_pair'],
      dtype='object')


statistic_yyy            object
according                object
site_id                  object
marriage_type            object
female_age_or_spouse1    object
male_age_or_spouse2      object
marry_pair                int64
dtype: object

In [24]:
# 列印出marriage_type有哪幾種種類的值

set(df.marriage_type)

{'不同性別', '相同性別_女性', '相同性別_男性'}

## 2. Summarize single variable

計算男同、女同、異性婚姻各有幾對。預期結果如下（數字可能不同）：
```
marriage_type
不同性別       112537
相同性別_女性      1323
相同性別_男性       536
Name: marry_pair, dtype: int64

```

In [28]:
# 計算男同、女同、異性婚姻各有幾對

df.groupby('marriage_type')['marry_pair'].sum()

marriage_type
不同性別       112537
相同性別_女性      1323
相同性別_男性       536
Name: marry_pair, dtype: int64

## 3. Summarize two variable hirarchically
計算每個鄉鎮市區的男同、女同、異性婚姻人數各有幾對. 你可能會需要謹慎觀看影片或者上網查詢「groupby two variables」。預期結果如下：
```
site_id  marriage_type
南投縣中寮鄉   不同性別              59
         相同性別_女性            0
         相同性別_男性            0
南投縣仁愛鄉   不同性別              84
         相同性別_女性            1
                         ... 
高雄市鹽埕區   相同性別_女性            1
         相同性別_男性            4
高雄市鼓山區   不同性別             716
         相同性別_女性           19
         相同性別_男性           12
Name: marry_pair, Length: 1104, dtype: int64
```

In [30]:
# 計算每個鄉鎮市區的男同、女同、異性婚姻人數各有幾對

df.groupby(['site_id', 'marriage_type'])['marry_pair'].sum()

site_id  marriage_type
南投縣中寮鄉   不同性別              45
         相同性別_女性            0
         相同性別_男性            0
南投縣仁愛鄉   不同性別              85
         相同性別_女性            1
                         ... 
高雄市鹽埕區   相同性別_女性            2
         相同性別_男性            2
高雄市鼓山區   不同性別             644
         相同性別_女性           11
         相同性別_男性            8
Name: marry_pair, Length: 1104, dtype: int64

## 4. Convert groupby result to dataframe

上述groupby的結果會變為series型態，請上網查詢如何把groupby的結果轉為dataframe。並把轉出來的dataframe指給變數`mstat`(Hints: `to_frame()`, `reset_index()`)。

預期輸出：
```
	site_id	marriage_type	marry_pair
0	南投縣中寮鄉	不同性別	59
1	南投縣中寮鄉	相同性別_女性	0
2	南投縣中寮鄉	相同性別_男性	0
3	南投縣仁愛鄉	不同性別	84
4	南投縣仁愛鄉	相同性別_女性	1
...	...	...	...
1099	高雄市鹽埕區	相同性別_女性	1
1100	高雄市鹽埕區	相同性別_男性	4
1101	高雄市鼓山區	不同性別	716
1102	高雄市鼓山區	相同性別_女性	19
1103	高雄市鼓山區	相同性別_男性	12
1104 rows × 3 columns
```

In [33]:
mstat = df.groupby(['site_id', 'marriage_type'])['marry_pair'].sum().to_frame().reset_index()
mstat

,site_id,marriage_type,marry_pair
0,南投縣中寮鄉,不同性別,45
1,南投縣中寮鄉,相同性別_女性,0
2,南投縣中寮鄉,相同性別_男性,0
3,南投縣仁愛鄉,不同性別,85
4,南投縣仁愛鄉,相同性別_女性,1
...,...,...,...
1099,高雄市鹽埕區,相同性別_女性,2
1100,高雄市鹽埕區,相同性別_男性,2
1101,高雄市鼓山區,不同性別,644
1102,高雄市鼓山區,相同性別_女性,11


## 5. Spread marriage_type to wide form

基於長表格（long table）的設計，目前結婚類型放在`marriage_type`這個欄位。現在我希望把`marriage_type`的數值（相同性別_女性、不同性別、相同性別_男性）展開在欄（Column）的方向，預計產生如下列型態的dataframe，方法可參考以下pandas對pivot的說明與範例。https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot.html

```
marriage_type	不同性別	相同性別_女性	相同性別_男性
site_id			
南投縣中寮鄉	59	0	0
南投縣仁愛鄉	84	1	0
南投縣信義鄉	97	0	0
南投縣南投市	551	4	1
南投縣名間鄉	200	0	0
...	...	...	...
高雄市阿蓮區	156	2	0
高雄市鳥松區	280	2	1
高雄市鳳山區	2078	36	24
高雄市鹽埕區	126	1	4
高雄市鼓山區	716	19	12
368 rows × 3 columns
```

In [34]:
mstat_wide = mstat.pivot(index = "site_id", columns='marriage_type', values='marry_pair')
print(type(mstat_wide))
mstat_wide

<class 'pandas.core.frame.DataFrame'>


marriage_type,不同性別,相同性別_女性,相同性別_男性
site_id,,,
南投縣中寮鄉,45,0,0
南投縣仁愛鄉,85,1,1
南投縣信義鄉,102,1,0
南投縣南投市,447,2,2
南投縣名間鄉,158,1,0
...,...,...,...
高雄市阿蓮區,120,0,0
高雄市鳥松區,219,1,1
高雄市鳳山區,1855,26,17


## 6. Using reset_index() to convert index to normal column(variable)

上題結果將會產生有階層的欄位，如下：
```
marriage_type    不同性別    相同性別_女性    相同性別_男性
site_id         
南投縣中寮鄉	59	0	0
...
```

如果希望把這樣的狀況轉為一般的dataframe（如下）可以用`reset_index()`這個函式。參考資料如下連結。https://datatofish.com/index-to-column-pandas-dataframe/ 請照上面網址中的範例嘗試看看，將前一題的`mstat_wide`轉為以下型態。
```
marriage_type	site_id	不同性別	相同性別_女性	相同性別_男性
0	南投縣中寮鄉	59	0	0
1	南投縣仁愛鄉	84	1	0
2	南投縣信義鄉	97	0	0
3	南投縣南投市	551	4	1
4	南投縣名間鄉	200	0	0
...	...	...	...	...
363	高雄市阿蓮區	156	2	0
364	高雄市鳥松區	280	2	1
365	高雄市鳳山區	2078	36	24
366	高雄市鹽埕區	126	1	4
367	高雄市鼓山區	716	19	12
```

In [35]:
mstat_wide.reset_index(inplace=True)

# Print the dataframe out
mstat_wide

marriage_type,site_id,不同性別,相同性別_女性,相同性別_男性
0,南投縣中寮鄉,45,0,0
1,南投縣仁愛鄉,85,1,1
2,南投縣信義鄉,102,1,0
3,南投縣南投市,447,2,2
4,南投縣名間鄉,158,1,0
...,...,...,...,...
363,高雄市阿蓮區,120,0,0
364,高雄市鳥松區,219,1,1
365,高雄市鳳山區,1855,26,17
366,高雄市鹽埕區,103,2,2


## 7. Rename variable names

重新修改變數名稱（欄位名稱）為`side_id`, `hetero`, `homoF`, `homoM`

In [50]:
mstat_wide.columns = ['side_id', 'hetero', 'homoF', 'homoM']
mstat_wide

,side_id,hetero,homoF,homoM
0,南投縣中寮鄉,59,0,0
1,南投縣仁愛鄉,84,1,0
2,南投縣信義鄉,97,0,0
3,南投縣南投市,551,4,1
4,南投縣名間鄉,200,0,0
...,...,...,...,...
363,高雄市阿蓮區,156,2,0
364,高雄市鳥松區,280,2,1
365,高雄市鳳山區,2078,36,24
366,高雄市鹽埕區,126,1,4


## 8. Mutate new variable

設計簡單的運算以產生每個鄉鎮的結婚總對數（`tot`）、同婚總對數（`homotot`）、同婚總對數所佔比例（`homoratio`），請依照題意指定變數名稱。
```
	side_id	hetero	homoF	homoM	homotot	tot	homoratio
0	南投縣中寮鄉	59	0	0	0	59	0.000000
1	南投縣仁愛鄉	84	1	0	1	85	0.011765
2	南投縣信義鄉	97	0	0	0	97	0.000000
3	南投縣南投市	551	4	1	5	556	0.008993
4	南投縣名間鄉	200	0	0	0	200	0.000000
...	...	...	...	...	...	...	...
363	高雄市阿蓮區	156	2	0	2	158	0.012658
364	高雄市鳥松區	280	2	1	3	283	0.010601
365	高雄市鳳山區	2078	36	24	60	2138	0.028064
366	高雄市鹽埕區	126	1	4	5	131	0.038168
367	高雄市鼓山區	716	19	12	31	747	0.041499
```

In [57]:
mstat_wide = mstat_wide.assign(homotot = mstat_wide.homoF + mstat_wide.homoM,
                               tot = mstat_wide.homoF + mstat_wide.homoM + mstat_wide.hetero)
mstat_wide

,side_id,hetero,homoF,homoM,homotot,tot
0,南投縣中寮鄉,59,0,0,0,59
1,南投縣仁愛鄉,84,1,0,1,85
2,南投縣信義鄉,97,0,0,0,97
3,南投縣南投市,551,4,1,5,556
4,南投縣名間鄉,200,0,0,0,200
...,...,...,...,...,...,...
363,高雄市阿蓮區,156,2,0,2,158
364,高雄市鳥松區,280,2,1,3,283
365,高雄市鳳山區,2078,36,24,60,2138
366,高雄市鹽埕區,126,1,4,5,131


In [59]:
mstat_wide = mstat_wide.assign(homoratio = mstat_wide.homotot/mstat_wide.tot)
mstat_wide

,side_id,hetero,homoF,homoM,homotot,tot,homoratio
0,南投縣中寮鄉,59,0,0,0,59,0.000000
1,南投縣仁愛鄉,84,1,0,1,85,0.011765
2,南投縣信義鄉,97,0,0,0,97,0.000000
3,南投縣南投市,551,4,1,5,556,0.008993
4,南投縣名間鄉,200,0,0,0,200,0.000000
...,...,...,...,...,...,...,...
363,高雄市阿蓮區,156,2,0,2,158,0.012658
364,高雄市鳥松區,280,2,1,3,283,0.010601
365,高雄市鳳山區,2078,36,24,60,2138,0.028064
366,高雄市鹽埕區,126,1,4,5,131,0.038168


## 9. Sorting

為了列印出同婚對數相較於總對數的比例最高的10個鄉鎮市區，請用`sort_values`依前題新產生的變數`homeratio`將該dataframe的列資料重新排序。
```
side_id	hetero	homoF	homoM	homotot	tot	homoratio
72	屏東縣獅子鄉	31	3	0	3	34	0.088235
74	屏東縣瑪家鄉	51	3	1	4	55	0.072727
274	花蓮縣新城鄉	101	5	2	7	108	0.064815
78	屏東縣車城鄉	31	1	1	2	33	0.060606
68	屏東縣泰武鄉	33	2	0	2	35	0.057143
272	花蓮縣壽豐鄉	83	5	0	5	88	0.056818
346	高雄市杉林區	53	2	1	3	56	0.053571
130	新北市烏來區	53	3	0	3	56	0.053571
341	高雄市左營區	993	27	25	52	1045	0.049761
277	花蓮縣秀林鄉	136	5	2	7	143	0.048951
```

In [65]:
mstat_wide.sort_values('homoratio', ascending=False)[:10]


,side_id,hetero,homoF,homoM,homotot,tot,homoratio
72,屏東縣獅子鄉,31,3,0,3,34,0.088235
74,屏東縣瑪家鄉,51,3,1,4,55,0.072727
274,花蓮縣新城鄉,101,5,2,7,108,0.064815
78,屏東縣車城鄉,31,1,1,2,33,0.060606
68,屏東縣泰武鄉,33,2,0,2,35,0.057143
272,花蓮縣壽豐鄉,83,5,0,5,88,0.056818
346,高雄市杉林區,53,2,1,3,56,0.053571
130,新北市烏來區,53,3,0,3,56,0.053571
341,高雄市左營區,993,27,25,52,1045,0.049761
277,花蓮縣秀林鄉,136,5,2,7,143,0.048951


## 10. Summarizing by hand

在下題中，老師希望你不要用pandas的方式來操作數據，希望你用一般的檔案讀取方法與for-loop來走訪所有資料。並建立一個以縣市為key的dictionary，來統計每個縣市，相同性別_男性、相同性別_女性、不同性別的結婚對數各有幾對。以下兩段程式碼會幫助你處理好資料。

1. 第一段程式碼是用`f.read()`來讀取資料，請注意觀察資料讀取後的格式。
2. 本段程式碼會需要讀取硬碟中的資料，

In [38]:
!wget https://raw.githubusercontent.com/p4css/py4css/main/data/opendata110m043.csv -O opendata110m043.csv
raw = []
with open("opendata110m043.csv", "r", encoding="utf-8-sig") as f:
    for line in f.read().split("\n"):
        row = line.split(",")
        raw.append(row)
print(raw[0])
print(raw[1])
print(raw[3])

--2022-10-10 16:06:30--  https://raw.githubusercontent.com/p4css/py4css/main/data/opendata110m043.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 56768523 (54M) [text/plain]
Saving to: 'opendata110m043.csv'

opendata110m043.csv 100%[===================>]  54.14M  41.6MB/s    in 1.3s    

2022-10-10 16:06:34 (41.6 MB/s) - 'opendata110m043.csv' saved [56768523/56768523]

['statistic_yyy', 'according', 'site_id', 'marriage_type', 'female_age_or_spouse1', 'male_age_or_spouse2', 'marry_pair']
['統計年度', '按照別', '區域別', '婚姻類型', '女方年齡或配偶一方年齡', '男方年齡或配偶另一方年齡', '結婚對數']
['110', '按發生日期分', '新北市板橋區', '不同性別', '未滿15歲', '15歲', '0']


### Retrieve county name

下面這段程式碼已經幫你寫好了，他會將原本的每一筆資料多產生一個變數（dictionary的key）是`county`，方法很簡單就只是將原本的縣市鄉鎮市區取出前三個字，如下面程式碼的`tdict['site_id'][:3]`。請勿修改以下程式碼，以下程式碼會
1. 將每筆資料的欄位對應到統一的欄位名稱、
2. 拆解出年齡區間並產生新變數`age_group`
3. 取得每筆資料的縣市名稱`county`
4. 移除不必要的變數

In [39]:
alldata = []
print(raw[0])
for i, row in enumerate(raw[2:-1]):
    tdict = dict(zip(raw[0], row))
    try:
        tdict['age_group'] = tdict['female_age_or_spouse1'] + "__" + tdict['male_age_or_spouse2']
    except:
        print(i, tdict)
    keys_to_remove = ['according', 'statistic_yyy', 'female_age_or_spouse1', 'male_age_or_spouse2']
    tdict['county'] = tdict['site_id'][:3]
    for k in keys_to_remove:
        tdict.pop(k, None)
    alldata.append(tdict)
print(alldata[:3])

['statistic_yyy', 'according', 'site_id', 'marriage_type', 'female_age_or_spouse1', 'male_age_or_spouse2', 'marry_pair']
[{'site_id': '新北市板橋區', 'marriage_type': '不同性別', 'marry_pair': '0', 'age_group': '未滿15歲__未滿15歲', 'county': '新北市'}, {'site_id': '新北市板橋區', 'marriage_type': '不同性別', 'marry_pair': '0', 'age_group': '未滿15歲__15歲', 'county': '新北市'}, {'site_id': '新北市板橋區', 'marriage_type': '不同性別', 'marry_pair': '0', 'age_group': '未滿15歲__16歲', 'county': '新北市'}]


### Summarize marriage_type by county

現在我希望用一個dict去統計，每個縣市（`county`）不同的婚姻類型（`marriage_type`）各有幾對。我的想法是用計數（Counting）的技巧，所以要建立兩層的dict來累計每個county有多少個不同marriage_type的婚姻案例。內層為marriage_type，外層為county。預期結果如下。

```json
{   '南投縣': {'不同性別': 2189, '相同性別_女性': 12, '相同性別_男性': 6},
    '嘉義市': {'不同性別': 1214, '相同性別_女性': 16, '相同性別_男性': 7},
    '嘉義縣': {'不同性別': 2071, '相同性別_女性': 11, '相同性別_男性': 4},
    '基隆市': {'不同性別': 1644, '相同性別_女性': 28, '相同性別_男性': 6},
    '宜蘭縣': {'不同性別': 2137, '相同性別_女性': 25, '相同性別_男性': 7},
    '屏東縣': {'不同性別': 3696, '相同性別_女性': 50, '相同性別_男性': 18},
    '彰化縣': {'不同性別': 5647, '相同性別_女性': 30, '相同性別_男性': 12},
    '新北市': {'不同性別': 18972, '相同性別_女性': 231, '相同性別_男性': 107},
    '新竹市': {'不同性別': 2331, '相同性別_女性': 36, '相同性別_男性': 5},
    '新竹縣': {'不同性別': 3016, '相同性別_女性': 36, '相同性別_男性': 10},
    '桃園市': {'不同性別': 12062, '相同性別_女性': 169, '相同性別_男性': 56},
    '澎湖縣': {'不同性別': 528, '相同性別_女性': 4, '相同性別_男性': 1},
    '臺中市': {'不同性別': 15155, '相同性別_女性': 167, '相同性別_男性': 69},
    '臺北市': {'不同性別': 11083, '相同性別_女性': 127, '相同性別_男性': 81},
    '臺南市': {'不同性別': 8775, '相同性別_女性': 111, '相同性別_男性': 29},
    '臺東縣': {'不同性別': 955, '相同性別_女性': 24, '相同性別_男性': 6},
    '花蓮縣': {'不同性別': 1720, '相同性別_女性': 36, '相同性別_男性': 9},
    '苗栗縣': {'不同性別': 2526, '相同性別_女性': 25, '相同性別_男性': 7},
    '連江縣': {'不同性別': 66, '相同性別_女性': 0, '相同性別_男性': 0},
    '金門縣': {'不同性別': 484, '相同性別_女性': 6, '相同性別_男性': 4},
    '雲林縣': {'不同性別': 2828, '相同性別_女性': 18, '相同性別_男性': 4},
    '高雄市': {'不同性別': 13438, '相同性別_女性': 161, '相同性別_男性': 88}}
```

In [43]:
mdict = {}
for row in alldata:
    if row['county'] not in mdict:
        mdict[row['county']] = {}
    if row['marriage_type'] not in mdict[row['county']]:
        mdict[row['county']][row['marriage_type']] = 0
    mdict[row['county']][row['marriage_type']] += int(row['marry_pair'])

    
# import pprint
# pp = pprint.PrettyPrinter(indent=4)
# pp.pprint(mdict)

{   '南投縣': {'不同性別': 2189, '相同性別_女性': 12, '相同性別_男性': 6},
    '嘉義市': {'不同性別': 1214, '相同性別_女性': 16, '相同性別_男性': 7},
    '嘉義縣': {'不同性別': 2071, '相同性別_女性': 11, '相同性別_男性': 4},
    '基隆市': {'不同性別': 1644, '相同性別_女性': 28, '相同性別_男性': 6},
    '宜蘭縣': {'不同性別': 2137, '相同性別_女性': 25, '相同性別_男性': 7},
    '屏東縣': {'不同性別': 3696, '相同性別_女性': 50, '相同性別_男性': 18},
    '彰化縣': {'不同性別': 5647, '相同性別_女性': 30, '相同性別_男性': 12},
    '新北市': {'不同性別': 18972, '相同性別_女性': 231, '相同性別_男性': 107},
    '新竹市': {'不同性別': 2331, '相同性別_女性': 36, '相同性別_男性': 5},
    '新竹縣': {'不同性別': 3016, '相同性別_女性': 36, '相同性別_男性': 10},
    '桃園市': {'不同性別': 12062, '相同性別_女性': 169, '相同性別_男性': 56},
    '澎湖縣': {'不同性別': 528, '相同性別_女性': 4, '相同性別_男性': 1},
    '臺中市': {'不同性別': 15155, '相同性別_女性': 167, '相同性別_男性': 69},
    '臺北市': {'不同性別': 11083, '相同性別_女性': 127, '相同性別_男性': 81},
    '臺南市': {'不同性別': 8775, '相同性別_女性': 111, '相同性別_男性': 29},
    '臺東縣': {'不同性別': 955, '相同性別_女性': 24, '相同性別_男性': 6},
    '花蓮縣': {'不同性別': 1720, '相同性別_女性': 36, '相同性別_男性': 9},
    '苗栗縣': {'不同性別': 2526, '相同性別_女

### 將上述結果用for-loop整齊列印出來。

預期結果：
```
county	不同性別	相同性別_男性	相同性別_女性
新北市	18972	107	231
臺北市	11083	81	127
桃園市	12062	56	169
臺中市	15155	69	167
臺南市	8775	29	111
高雄市	13438	88	161
宜蘭縣	2137	7	25
新竹縣	3016	10	36
苗栗縣	2526	7	25
彰化縣	5647	12	30
南投縣	2189	6	12
雲林縣	2828	4	18
嘉義縣	2071	4	11
屏東縣	3696	18	50
臺東縣	955	6	24
花蓮縣	1720	9	36
澎湖縣	528	1	4
基隆市	1644	6	28
新竹市	2331	5	36
嘉義市	1214	7	16
金門縣	484	4	6
連江縣	66	0	0


```

In [44]:
print("county\t%s"%('\t'.join(mdict['新北市'].keys())))
for k, v in mdict.items():
    print("%s\t%s\t%s\t%s"%(k, v['不同性別'], v['相同性別_男性'], v['相同性別_女性']))


county	不同性別	相同性別_男性	相同性別_女性
新北市	18972	107	231
臺北市	11083	81	127
桃園市	12062	56	169
臺中市	15155	69	167
臺南市	8775	29	111
高雄市	13438	88	161
宜蘭縣	2137	7	25
新竹縣	3016	10	36
苗栗縣	2526	7	25
彰化縣	5647	12	30
南投縣	2189	6	12
雲林縣	2828	4	18
嘉義縣	2071	4	11
屏東縣	3696	18	50
臺東縣	955	6	24
花蓮縣	1720	9	36
澎湖縣	528	1	4
基隆市	1644	6	28
新竹市	2331	5	36
嘉義市	1214	7	16
金門縣	484	4	6
連江縣	66	0	0
